# Tone and Register Modes

Tone and register modes shape *how* an agent communicates — its level of formality, technical depth, sentence structure, and stylistic register. While persona modes define *who* the agent is and domain constraint modes define *where* it operates, tone and register modes define the language the agent uses to express itself within that identity and scope.

Register is not a style preference — it is a communication requirement determined by the audience, context, stakes, and channel of a conversation. A post-incident report read by an executive needs a different opening than the same report read by the engineering team. A customer support response to a non-technical user needs different vocabulary than a response to a developer. A legal filing needs different sentence structure than a Slack message.

The right register makes content accessible to the right audience. The wrong register loses readers, undermines trust, or creates confusion — even when the underlying information is correct. An executive who receives a dense technical root-cause analysis instead of a three-sentence decision summary will stop reading before reaching the information they need.

## The register spectrum

```
FORMAL / TECHNICAL                              CASUAL / EXPLORATORY
--------------------------------------------------------------------
Legal      Academic    Executive   Professional  Conversational  Casual
Prose       Paper       Brief       Email         Chat            Slang
  |           |           |            |              |             |
Maximum    High        Structured   Balanced     Informal       Minimal
Precision  rigor       concision    clarity      warm           formality
--------------------------------------------------------------------
```

This notebook implements four registers that cover the practical range of agent use cases: **formal/technical**, **executive**, **professional**, and **conversational**. Each is encoded as a structured configuration, translated into a system prompt, and demonstrated on the same content so the differences are immediately visible.

In [1]:
import os
from dataclasses import dataclass
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

### Initialize the language model
Register modes use `temperature=0` to ensure that the same content expressed in the same register produces consistent output across runs. An executive briefing should always lead with the recommendation; a formal technical document should always use the same structural conventions. Temperature-driven variation would undermine the reliability that register modes are designed to provide.

In [2]:
# temperature=0 for consistent, predictable register application
llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.getenv("OPENAI_API_KEY", "").strip(),
    temperature=0,
)

print("LLM initialized.")

LLM initialized.


## The simplest register: a system prompt instruction
Before introducing any structure or classes, it is worth seeing what a register actually is at the implementation level. A register mode is a system prompt instruction that shapes the agent's communication style. The same content request, sent with different system messages, produces outputs that differ in vocabulary, structure, sentence length, and formality — even when the underlying information is identical.

The example below sends the same request — "Explain what an API is" — three times: once with no system instruction, once with a formal instruction, and once with a conversational instruction. This is the irreducible core of register modes.

In [3]:
request = "Explain what an API is."

# No register instruction -- the LLM uses its own default style
no_register = llm.invoke([
    HumanMessage(content=request),
]).content

# Formal register -- precision, no contractions, structured language
formal_register = llm.invoke([
    SystemMessage(content=(
        "Communicate in a formal, technical register. "
        "Use precise domain terminology. Avoid contractions. "
        "Use passive voice where appropriate. Structure your response with clear sections."
    )),
    HumanMessage(content=request),
]).content

# Conversational register -- warm, informal, short sentences
conversational_register = llm.invoke([
    SystemMessage(content=(
        "Communicate in a warm, conversational register. "
        "Use everyday language. Contractions are fine. "
        "Keep sentences short. Match a supportive, friendly tone."
    )),
    HumanMessage(content=request),
]).content

print("NO REGISTER INSTRUCTION:")
print("-" * 60)
print(no_register)
print()
print("FORMAL REGISTER:")
print("-" * 60)
print(formal_register)
print()
print("CONVERSATIONAL REGISTER:")
print("-" * 60)
print(conversational_register)

NO REGISTER INSTRUCTION:
------------------------------------------------------------
An API, or Application Programming Interface, is a set of rules and protocols that allows different software applications to communicate with each other. It defines the methods and data formats that applications can use to request and exchange information. APIs enable developers to access the functionality of other software components, services, or platforms without needing to understand their internal workings.

Here are some key points about APIs:

1. **Interoperability**: APIs allow different systems, applications, or services to work together, facilitating data exchange and functionality sharing.

2. **Abstraction**: They provide a simplified interface to complex systems, allowing developers to use predefined functions without needing to know the underlying code or architecture.

3. **Types of APIs**:
   - **Web APIs**: These are accessed over the internet using HTTP/HTTPS protocols. They are comm

The three responses address the same question but differ in every structural dimension: vocabulary choice, sentence length, use of contractions, use of technical terms, and whether the response leads with a definition or with a conversational acknowledgment. This difference comes entirely from the system prompt.

This simple approach works for a single register. But when an application needs four registers — and needs them applied consistently across multiple agents and sessions — a structured configuration is more appropriate. The configuration approach also enables an important practice: including a concrete example of the register in the prompt. Abstract instructions like "be formal" are interpreted variably; a sample sentence gives the model a specific target to match.

## Structuring a register as configuration
Each register is defined by six dimensions that together fully specify how the agent should communicate. The `vocabulary` and `sentence_length` fields govern word choice and phrasing rhythm. The `structure` field describes organizational conventions — an executive register leads with the recommendation; a formal technical register uses a hierarchical section structure. The `contractions` and `first_person` fields are formality signals: "we're" is informal; "we are" is formal. The `hedging_style` governs how uncertainty is expressed — precise confidence intervals for technical documents, natural "I think" for conversational exchanges. The `example` field is the most important: it gives the model a concrete sample of the target register rather than only abstract descriptions.

In [4]:
# Each register is a dictionary of six communication dimensions plus metadata.
# The 'example' field is deliberately included -- concrete samples are more reliable than abstract instructions like 'be formal' which the model interprets variably.

FORMAL_TECHNICAL_REGISTER = {
    "name": "formal_technical",
    "display_name": "Formal / Technical",
    "vocabulary": "domain-specific, precise, no abbreviations without definition",
    "sentence_length": "medium to long -- completeness over brevity",
    # Hierarchical structure signals a formal, document-like output
    "structure": "hierarchical: sections, subsections, numbered lists",
    "contractions": False,       # 'we are' not 'we're'
    "first_person": False,       # 'the analysis demonstrates' not 'I found'
    "hedging_style": "explicit confidence levels and qualifications (e.g., 'with 95% confidence')",
    "best_for": ["technical specifications", "academic papers", "regulatory filings"],
    # A concrete example sentence anchors the model to a specific target style
    "example": (
        "The assessment indicates a statistically significant correlation "
        "(r = 0.82, p < 0.01) between latency variance and user session "
        "abandonment rates during peak traffic periods."
    ),
}

EXECUTIVE_REGISTER = {
    "name": "executive",
    "display_name": "Executive",
    "vocabulary": "business-oriented, outcome-focused, no unexplained jargon",
    # One idea per sentence -- executives scan, they do not read linearly
    "sentence_length": "short -- one idea per sentence",
    # Recommendation first is the defining structural rule of executive register
    "structure": "recommendation first, then rationale, then supporting data",
    "contractions": False,
    "first_person": "sparingly",
    # Minimal hedging: executives need a clear position, not a list of caveats
    "hedging_style": "minimal -- lead with the recommendation, caveat briefly at the end",
    "best_for": ["executive briefings", "board presentations", "C-suite recommendations"],
    "example": (
        "Recommendation: Migrate to the new authentication system by Q3. "
        "Current system has 3 critical vulnerabilities. "
        "Migration cost: $120K. Risk of breach at current rate: $2.4M/year."
    ),
}

PROFESSIONAL_REGISTER = {
    "name": "professional",
    "display_name": "Professional",
    "vocabulary": "clear, specific, no unnecessary jargon",
    # Sentence variety makes professional writing readable without feeling casual
    "sentence_length": "varied -- mix of short and medium sentences",
    "structure": "key point first, then supporting details",
    "contractions": True,        # 'we're', 'it's' are appropriate in business writing
    "first_person": True,
    "hedging_style": "honest uncertainty acknowledged briefly",
    "best_for": ["business emails", "team documentation", "product documentation"],
    "example": (
        "I've reviewed the API performance data for Q4. Latency is up 40% from Q3, "
        "primarily driven by the new authentication middleware. "
        "I recommend we profile the middleware before the next release."
    ),
}

CONVERSATIONAL_REGISTER = {
    "name": "conversational",
    "display_name": "Conversational",
    # Only use jargon if the user introduced it first
    "vocabulary": "everyday language, avoid jargon unless the user uses it first",
    "sentence_length": "short -- conversational rhythm",
    # Follow the flow of the exchange rather than imposing a document structure
    "structure": "follow the conversation flow, not a rigid template",
    "contractions": True,
    "first_person": True,
    "hedging_style": "natural: 'I think', 'probably', 'it depends'",
    "best_for": ["customer support", "user-facing chatbots", "tutoring", "casual advisory"],
    "example": (
        "Good question! The short answer is yes, but it depends on your setup. "
        "If you're on the free tier, you'll hit the rate limit pretty quickly. "
        "Want me to walk you through upgrading?"
    ),
}

ALL_REGISTERS = [
    FORMAL_TECHNICAL_REGISTER,
    EXECUTIVE_REGISTER,
    PROFESSIONAL_REGISTER,
    CONVERSATIONAL_REGISTER,
]

print("Register configurations defined:")
for r in ALL_REGISTERS:
    best = r["best_for"][0]  # show the primary use case
    print(f"  {r['display_name']}: best for {best}")

Register configurations defined:
  Formal / Technical: best for technical specifications
  Executive: best for executive briefings
  Professional: best for business emails
  Conversational: best for customer support


Four register configurations with a consistent six-field schema. The comments inside the configs explain the reasoning behind each design decision — why `sentence_length` for the executive register is "short" (executives scan, they do not read linearly), why `first_person` for formal/technical is `False` (passive voice signals analytical objectivity). These design choices translate directly into system prompt instructions.

## Translating configuration to a system prompt
With registers defined as dictionaries, the next step is translating them into system prompt strings. The `RegisterMode` class wraps a configuration and exposes prompt-building as a method. Like the `PersonaMode` class from the previous notebook, it is kept focused on prompt generation — it does not make any LLM calls. This keeps it testable and model-agnostic.

The prompt template uses a tabular layout — `Vocabulary:`, `Sentence length:` — to give the model clearly labeled instructions. The `example` field is placed last and formatted as a quoted sentence, which signals to the model that it is a sample to match rather than an instruction to follow.

In [5]:
class RegisterMode:
    """Wraps a register configuration and generates system prompt components from it.

    Intentionally does not make LLM calls -- prompt building and model invocation are kept separate so register modes are testable without API access.
    """

    def __init__(self, config: dict):
        self.config = config
        self.name = config["name"]
        self.display_name = config["display_name"]

    def build_prompt(self) -> str:
        """Generate the system prompt component that activates this register.

        Returns:
            A formatted string ready for inclusion in a SystemMessage.
        """
        # Convert boolean contractions/first_person fields to readable instructions
        contractions_str = "Use freely" if self.config["contractions"] else "Avoid"
        fp = self.config["first_person"]
        first_person_str = (
            "Use when natural" if fp is True
            else "Avoid -- prefer passive or third person" if fp is False
            else str(fp)  # handles the 'sparingly' string for executive register
        )

        return (
            f"COMMUNICATION REGISTER: {self.display_name.upper()}\n\n"
            f"Vocabulary     : {self.config['vocabulary']}\n"
            f"Sentence length: {self.config['sentence_length']}\n"
            f"Structure      : {self.config['structure']}\n"
            f"Contractions   : {contractions_str}\n"
            f"First person   : {first_person_str}\n"
            f"Uncertainty    : {self.config['hedging_style']}\n\n"
            # The quoted example gives the model a concrete target to match
            f"Example of this register:\n\"{self.config['example']}\""
        )


# Instantiate each register mode by name -- these objects are reused across all demos
formal = RegisterMode(FORMAL_TECHNICAL_REGISTER)
executive = RegisterMode(EXECUTIVE_REGISTER)
professional = RegisterMode(PROFESSIONAL_REGISTER)
conversational = RegisterMode(CONVERSATIONAL_REGISTER)

# Registry for lookup by string key -- used by the adaptive agent
REGISTER_MAP = {
    "formal_technical": formal,
    "executive": executive,
    "professional": professional,
    "conversational": conversational,
}

print("RegisterMode instances created.")
print("\nSystem prompt for Professional register:")
print("-" * 60)
print(professional.build_prompt())

RegisterMode instances created.

System prompt for Professional register:
------------------------------------------------------------
COMMUNICATION REGISTER: PROFESSIONAL

Vocabulary     : clear, specific, no unnecessary jargon
Sentence length: varied -- mix of short and medium sentences
Structure      : key point first, then supporting details
Contractions   : Use freely
First person   : Use when natural
Uncertainty    : honest uncertainty acknowledged briefly

Example of this register:
"I've reviewed the API performance data for Q4. Latency is up 40% from Q3, primarily driven by the new authentication middleware. I recommend we profile the middleware before the next release."


The `build_prompt()` method converts the configuration dictionary into a labeled system prompt. Boolean fields (`contractions`, `first_person`) are mapped to readable strings. The example sentence is quoted and placed at the end where it functions as a concrete anchor.

## Demo 1: Same content, four registers
The most direct way to understand register is to see the same information expressed in four different registers side by side. The finding below — about API latency degradation — is identical across all four calls. Only the system prompt changes. Notice how the register affects not just tone but structure, what information is foregrounded, and even what gets included at all.

In [6]:
# The same finding is passed to all four registers -- the content is identical,
# only the system prompt (the register) changes between calls.
CONTENT_REQUEST = """Write a brief communication about the following finding:

Finding: API response time has increased 40% over the past month.
Root cause: New authentication middleware in v2.3 adds 150ms overhead per request
due to an uncached token validation step.
Impact: P95 latency is now 820ms, up from 580ms. User session abandonment up 12%.
Fix: Add Redis caching to token validation. Estimated: 2 days engineering work.
"""

# Iterate over the four registers in order from most to least formal
for register in [formal, executive, professional, conversational]:
    print(f"{'=' * 60}")
    print(f"REGISTER: {register.display_name}")
    print(f"Best for: {', '.join(register.config['best_for'][:2])}")
    print("-" * 60)

    # Build the system prompt from the register configuration
    # and invoke the LLM with the same content request each time
    response = llm.invoke([
        SystemMessage(content=register.build_prompt()),
        HumanMessage(content=CONTENT_REQUEST),
    ]).content

    print(response)
    print()

REGISTER: Formal / Technical
Best for: technical specifications, academic papers
------------------------------------------------------------
Subject: Increased API Response Time and Proposed Mitigation Strategy

1. **Overview of Findings**
   The analysis of the application programming interface (API) performance has revealed a significant increase in response time, quantified at 40% over the past month. 

2. **Root Cause Analysis**
   The primary contributor to this increase in latency has been identified as the introduction of new authentication middleware in version 2.3. This middleware introduces an additional overhead of 150 milliseconds per request, attributed to an uncached token validation step.

3. **Impact Assessment**
   The increase in response time has resulted in a rise in the 95th percentile (P95) latency, which has escalated from 580 milliseconds to 820 milliseconds. Consequently, there has been a 12% increase in user session abandonment rates, indicating a potential d

The structural differences are as significant as the tonal ones. The formal/technical response uses a hierarchical structure, passive voice, and precise quantified claims. The executive response leads with the recommendation and a dollar figure — the data follows the decision, not the other way around. The professional response uses active voice and balanced detail accessible to both technical and non-technical colleagues. The conversational response leads with acknowledgment, uses plain language, and ends with a question to invite engagement.

The same engineering finding produces four substantively different communications — not because the information changed, but because the structural and vocabulary rules of each register reorganize and reframe it.

## Adaptive register detection
In many contexts the appropriate register is not fixed at deployment — it should adapt to the individual user's communication style. A user writing in casual shorthand should receive a conversational response; a user writing in technical prose should receive a technical response. Mismatching register undermines communication even when the information is correct.

The `detect_register()` function classifies a user message into one of the four registers by making a small, focused LLM call. Extracting this as a standalone function — rather than a private method on a class — makes it independently testable and reusable across different agent architectures.

In [7]:
def detect_register(llm, message: str) -> str:
    """Classify the communication register of a user message.

    Makes a small dedicated LLM call focused solely on style classification,
    separate from the main response generation.

    Args:
        llm: The language model to use for classification.
        message: The user message to classify.

    Returns:
        One of: 'formal_technical', 'executive', 'professional', 'conversational'.
    """
    classification_prompt = f"""Analyze the communication style of this message and classify its register.

Message: \"{message}\"

Register options:
- formal_technical: technical jargon, full sentences, no contractions, precise qualifications
- executive: brief, outcome-focused, business language, bottom-line first
- professional: clear and direct, moderate formality, active voice, complete sentences
- conversational: informal, warm, contractions, short sentences, casual phrasing

Respond with only the register name. One of: formal_technical, executive, professional, conversational"""

    response = llm.invoke([HumanMessage(content=classification_prompt)]).content.strip().lower()

    # Default to 'professional' if the model returns an unexpected value
    return response if response in REGISTER_MAP else "professional"


@dataclass
class AdaptiveRegisterAgent:
    """An agent that detects and adapts to the user's communication register."""

    llm: object
    default_register: str = "professional"  # starting register before any detection
    system_context: str = ""               # optional domain or persona context

    def __post_init__(self):
        # Initialize the current register from the default -- updates on each adapted turn
        self.current_register = REGISTER_MAP[self.default_register]

    def respond(self, user_message: str, adapt: bool = True) -> str:
        """Generate a response, optionally adapting the register to the user's style.

        Args:
            user_message: The incoming message from the user.
            adapt: When True, classify the user's message and update the register
                   if a shift is detected. When False, use the current register.

        Returns:
            A response formatted in the appropriate register.
        """
        if adapt:
            # Classify the user's message and switch registers if their style changed
            detected = detect_register(self.llm, user_message)
            if detected != self.current_register.name:
                print(f"  [Register shift: {self.current_register.display_name} -> {REGISTER_MAP[detected].display_name}]")
                self.current_register = REGISTER_MAP[detected]

        # Combine register prompt with any domain/persona context
        register_prompt = self.current_register.build_prompt()
        system_prompt = (
            f"{self.system_context}\n\n{register_prompt}"
            if self.system_context
            else register_prompt
        )

        return self.llm.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_message),
        ]).content


# Test detect_register directly before using it inside the agent
test_messages = [
    "hey, can you explain what a REST API is? i'm kinda new to this stuff",
    "What is the business case for adopting microservices? We need a decision by EOQ.",
    "Please delineate the architectural trade-offs between REST and GraphQL with respect to query efficiency.",
]

print("Register detection results:")
for msg in test_messages:
    detected = detect_register(llm, msg)
    # Trim the message for display to keep output readable
    print(f"  '{msg[:55]}...' -> {detected}")

Register detection results:
  'hey, can you explain what a REST API is? i'm kinda new ...' -> conversational
  'What is the business case for adopting microservices? W...' -> executive
  'Please delineate the architectural trade-offs between R...' -> formal_technical


`detect_register()` takes a message and returns a string key. The three test messages classify as conversational, executive, and formal/technical respectively — matching the vocabulary, phrasing, and structure signals in each message. The `AdaptiveRegisterAgent` uses this function on each turn and updates `self.current_register` whenever the detected register differs from the current one, printing a shift notification so the behavior is observable.

## Demo 2: Adaptive detection across conversation turns
The adaptive agent starts at the default professional register. Each message below is written in a distinctly different style, spanning the full register spectrum from conversational to executive to formal/technical. The agent should detect each shift and respond in a matching register without any explicit instruction from the user.

In [8]:
# The agent starts at 'professional' by default
agent = AdaptiveRegisterAgent(llm=llm, default_register="professional")

# Each message is written in a distinct register -- conversational, executive, formal/technical
user_messages = [
    "hey, can you give me a quick rundown of what a REST API is? i'm pretty new to this",
    "What is the business case for microservices? We need a board-level decision by Q3.",
    "Please provide a technical comparison of REST and GraphQL with respect to "
    "query efficiency, latency characteristics, and schema evolution patterns.",
]

print("Adaptive register detection across turns:")
print("=" * 60)

for i, message in enumerate(user_messages, 1):
    print(f"\nTurn {i}")
    # Display a truncated version of the user message for readability
    print(f"User: '{message[:70]}{'...' if len(message) > 70 else "'"}'")

    response = agent.respond(message)

    # Show which register the agent used after detection
    print(f"Register used: {agent.current_register.display_name}")
    print("-" * 60)
    print(response)
    print()

Adaptive register detection across turns:

Turn 1
User: 'hey, can you give me a quick rundown of what a REST API is? i'm pretty...'
  [Register shift: Professional -> Conversational]
Register used: Conversational
------------------------------------------------------------
Sure thing! A REST API, or Representational State Transfer Application Programming Interface, is a way for different software applications to communicate over the web. 

Basically, it uses standard HTTP methods like GET, POST, PUT, and DELETE to perform operations on resources, which are usually represented in formats like JSON or XML. 

So, if you wanted to get some data from a server, you'd use a GET request. If you wanted to send data, you'd use POST. It’s all about making it easy for different systems to talk to each other. Does that help?


Turn 2
User: 'What is the business case for microservices? We need a board-level dec...'
  [Register shift: Conversational -> Executive]
Register used: Executive
------------

The register shift notifications in the output show the agent tracking the user's communication style changes turn by turn. The first response mirrors the casual conversational tone; the second shifts to a concise, decision-oriented executive style; the third shifts again to a formal technical comparison. The user did not instruct any of these changes — the agent inferred them from the message content.

## Technical depth calibration
Register and technical depth are related but separate dimensions. A conversational register does not require low technical depth — a developer asking a casual question might still need a technically precise answer. An executive register does not require high technical depth — it uses simple vocabulary regardless of the underlying complexity.

The table below shows the typical pairing, but the dimensions are orthogonal and can be configured independently:

| Register | Typical depth | But can be |
|---|---|---|
| Formal / Technical | High | Low, for introductory technical documents |
| Executive | Low | High, for technically sophisticated executives |
| Professional | Calibrated to audience | Either, depending on the team |
| Conversational | User-adaptive | Very high, for developer-facing support |

The system prompt below demonstrates depth calibration: the same conversational register is paired with instructions to read the user's messages for expertise signals and adjust accordingly. Both example questions below use a conversational register — the depth adapts to what the user's phrasing reveals about their background.

In [9]:
# Depth calibration instructions are a separate system prompt component,
# independent of the register. They can be combined with any register.
DEPTH_CALIBRATION = """
TECHNICAL DEPTH CALIBRATION:

Read the user's vocabulary and question type to calibrate your depth:
- High expertise signals: precise terminology (e.g. 'TTFB', 'CAP theorem', 'O(log n)'),
  references to specific tools or standards, implementation-level questions.
- Low expertise signals: vague terms ('make it faster', 'that database thing'),
  conceptual questions without specifics, expressed confusion about terminology.

If HIGH expertise: use domain terminology, skip basic definitions, go deep.
If LOW expertise: define terms, use analogies, avoid unexplained jargon.
If UNKNOWN: start at a mid level and adjust on the next turn.
"""

# Combine the conversational register prompt with the depth calibration instructions
combined_prompt = conversational.build_prompt() + DEPTH_CALIBRATION

questions = [
    # General phrasing -- signals lower expertise; expects an accessible answer
    "How do I make my website load faster?",
    # Technical phrasing -- signals high expertise; expects a precise technical comparison
    "What are the trade-offs between HTTP/2 server push and service worker "
    "prefetching for reducing TTFB in a CDN-backed SPA?",
]

labels = ["Low expertise (conversational register)", "High expertise (conversational register)"]

for label, question in zip(labels, questions):
    print(f"\n{label}")
    print(f"User: {question}")
    print("-" * 60)

    # The same combined prompt handles both questions -- depth adapts automatically
    response = llm.invoke([
        SystemMessage(content=combined_prompt),
        HumanMessage(content=question),
    ]).content

    print(response)


Low expertise (conversational register)
User: How do I make my website load faster?
------------------------------------------------------------
Great question! There are a few key things you can do to speed up your website. Here are some tips:

1. **Optimize Images**: Make sure your images are the right size and compressed. Tools like TinyPNG can help with that.

2. **Minimize HTTP Requests**: Try to reduce the number of elements on your page. Fewer images, scripts, and stylesheets mean fewer requests.

3. **Use a Content Delivery Network (CDN)**: CDNs can help deliver your content faster by using servers that are closer to your users.

4. **Enable Browser Caching**: This allows visitors to store some data on their devices, so they don’t have to download it again on their next visit.

5. **Minify CSS and JavaScript**: Remove unnecessary characters from your code to make it smaller and faster to load.

6. **Choose a Good Hosting Provider**: Sometimes, the server speed can be a bottlen

Both responses use a conversational register — informal, warm, contractions — but the first uses analogies and everyday vocabulary while the second uses technical terminology without definition. The register is consistent across both; the depth adapts to what the user's phrasing reveals about their expertise level. This shows that the two dimensions can be controlled independently in the system prompt.

## Register consistency
A common failure mode is register drift: starting in one register and gradually shifting toward the user's style across turns. Drift happens because the model responds to conversational pressure — a casual follow-up message pulls the model's style toward casual phrasing even when the session was established as formal.

The consistency instruction below specifies not just what register to maintain, but the exact conditions under which a shift is legitimate. "Only shift register when the user explicitly requests it or the context fundamentally changes" is more enforceable than "maintain this register when appropriate" — which the model interprets as permission to drift whenever the turn feels right.

In [10]:
def build_consistency_prompt(register: RegisterMode) -> str:
    """Build a register consistency enforcement instruction.

    Specifies two explicit conditions for register change rather than the vague 'when appropriate' that models interpret as permission to drift.

    Args:
        register: The RegisterMode to enforce consistently.

    Returns:
        A system prompt component for session-level consistency.
    """
    return (
        f"REGISTER CONSISTENCY\n\n"
        f"You have established a {register.display_name} register in this session. "
        f"Maintain this register throughout, even when:\n"
        f"- The topic becomes more or less technical\n"
        f"- The conversation extends over many turns\n"
        f"- The user sends a brief, casual, or off-topic message\n\n"
        f"Only shift register when:\n"
        f"1. The user explicitly requests a different style (e.g. 'be more casual', 'write this formally')\n"
        f"2. The context fundamentally changes (e.g. from strategic planning to an urgent incident)\n\n"
        f"In all other cases, maintain {register.display_name} register consistently."
    )


# Combine the formal register with a consistency enforcement instruction
formal_session_prompt = formal.build_prompt() + "\n\n" + build_consistency_prompt(formal)

# The conversation starts formally but the second turn is deliberately casual
# to test whether the consistency instruction holds against conversational pressure
conversation = [
    "Provide an analysis of microservices vs monolithic architecture trade-offs.",
    # Casual follow-up -- without consistency enforcement this would trigger a style shift
    "ok but like, what would you actually recommend for a startup?",
]

# Maintain a proper message list: SystemMessage for context, then alternating Human/AI
messages = [SystemMessage(content=formal_session_prompt)]

print("Formal register consistency test:")
print("(Note: the casual follow-up should still receive a formal response)")
print("=" * 60)

for turn_num, user_msg in enumerate(conversation, 1):
    messages.append(HumanMessage(content=user_msg))
    response = llm.invoke(messages).content

    # Append the LLM response as AIMessage to maintain proper conversation role structure.
    # Using AIMessage (not SystemMessage) correctly models the assistant's turn.
    messages.append(AIMessage(content=response))

    print(f"\nTurn {turn_num}")
    print(f"User : {user_msg}")
    print(f"Agent: {response[:400]}..." if len(response) > 400 else f"Agent: {response}")

Formal register consistency test:
(Note: the casual follow-up should still receive a formal response)

Turn 1
User : Provide an analysis of microservices vs monolithic architecture trade-offs.
Agent: **Analysis of Microservices vs. Monolithic Architecture Trade-offs**

1. **Introduction**
   The architectural design of software systems significantly influences their scalability, maintainability, and deployment. This analysis delineates the trade-offs between microservices architecture and monolithic architecture, focusing on various dimensions such as scalability, development complexity, deplo...

Turn 2
User : ok but like, what would you actually recommend for a startup?
Agent: **Recommendation for Architectural Approach in a Startup Context**

1. **Introduction**
   The architectural choice for a startup is critical, as it can significantly influence the speed of development, scalability, and overall success of the product. This recommendation evaluates the suitability of monolithic 

The second response — the reply to "ok but like, what would you actually recommend" — should remain in the formal/technical register despite the casual phrasing of the user's follow-up. The consistency instruction explicitly says not to shift register based on a casual message; only an explicit user request or a fundamental context change justifies a shift.

**Important technical note:** The conversation loop appends the LLM's response as an `AIMessage`, not a `SystemMessage`. Using `AIMessage` correctly reflects the role structure of the conversation and ensures the LLM processes the history accurately. This is a common mistake in multi-turn demos that can subtly affect how the model interprets its own prior turns.

## Demo 3: Multi-audience output
Some outputs must serve multiple audiences simultaneously. A post-incident report is read by executives who need a decision summary, by engineers who need the full technical root cause, and by a broader team who needs accessible context. No single register serves all three audiences well.

The solution is structured output: different sections of the document are written in different registers, each targeted at its audience. Each reader goes directly to the section relevant to them — executives read the summary, engineers read the technical analysis — without having to parse content optimized for a different reader.

In [11]:
INCIDENT_DETAILS = """Incident: Production API outage on 2024-01-15, 14:23-16:47 UTC (2h 24min).
Root cause: Version 3.1.2 introduced a memory leak in the connection pool manager.
Under sustained load, heap exhaustion caused continuous GC cycles consuming 99% CPU.
Impact: 100% of API requests failed for 2h 24min. ~14,000 users affected. ~$87K revenue loss.
Resolution: Rolled back to 3.1.1 at 16:47 UTC. Recovery confirmed 16:52 UTC.
Fix: Connection pool manager rewritten with bounded memory allocation. Tests added.
3.1.3 scheduled after 48-hour staging validation.
"""

# The prompt specifies three sections with explicit register instructions for each.
# This is more reliable than a vague 'write for multiple audiences' instruction.
multi_audience_prompt = f"""Write a structured incident report for three audiences.

Incident details:
{INCIDENT_DETAILS}

Produce exactly three sections with these register requirements:

## Executive Summary
Register: EXECUTIVE -- 3-4 sentences maximum. Status and impact first, then recovery.
No technical jargon. Key numbers only (duration, users affected, revenue impact).

## Incident Overview
Register: PROFESSIONAL -- 1-2 paragraphs. Timeline, cause, and resolution in
clear business-accessible language. Readable by both technical and non-technical staff.

## Technical Root Cause Analysis
Register: FORMAL/TECHNICAL -- Full technical detail. Precise description of the
failure mode, diagnostic indicators, fix implementation, and preventive measures.
"""

response = llm.invoke([HumanMessage(content=multi_audience_prompt)]).content

print("Multi-audience incident report:")
print("=" * 60)
print(response)

Multi-audience incident report:
## Executive Summary
On January 15, 2024, from 14:23 to 16:47 UTC, our production API experienced a significant outage lasting 2 hours and 24 minutes, affecting approximately 14,000 users and resulting in an estimated revenue loss of $87,000. The issue was caused by a memory leak introduced in version 3.1.2. The service was restored by rolling back to version 3.1.1, with recovery confirmed shortly thereafter.

## Incident Overview
The production API outage occurred on January 15, 2024, beginning at 14:23 UTC and lasting until 16:47 UTC. During this period, all API requests failed, impacting around 14,000 users and leading to an estimated revenue loss of $87,000. The root cause was identified as a memory leak in the connection pool manager introduced with the deployment of version 3.1.2. Under sustained load, this leak resulted in heap exhaustion, causing the system to enter continuous garbage collection cycles that consumed 99% of CPU resources. To resol

Three sections, each in a distinct register, all describing the same incident. The executive summary leads with status and business impact in plain language. The incident overview uses active voice and professional vocabulary accessible across the organization. The technical root cause section uses precise terminology — memory leak, GC cycles, heap exhaustion — without any obligation to define these terms for a non-technical reader.

This is the core value of explicit register control in structured documents: each audience receives content calibrated to their vocabulary and needs, all within a single output.

- **Register is a system prompt instruction.** At its core, a register mode is a system message that describes vocabulary, sentence structure, formality level, and a concrete example sentence. The `RegisterMode` class and configurations are infrastructure for managing these instructions consistently across an application.
- **Register and technical depth are orthogonal dimensions.** A conversational register can accommodate high technical depth; a formal register does not always mean deep technical content. The `DEPTH_CALIBRATION` prompt component shows how to control depth independently of formality.
- **Register consistency must be explicitly enforced.** Without a consistency instruction, the model drifts toward the user's conversational style across turns. The instruction must specify the exact conditions for a legitimate shift — not just "maintain this register when appropriate."
- **Multi-audience documents need per-section register instructions.** When a single document must serve multiple audiences, assign an explicit register to each section. Readers navigate to the section calibrated to their vocabulary and skip the rest.

### Composing with other mode dimensions

| Combination | Result |
|---|---|
| Persona + Register | A legal advisor persona in formal/technical register produces precise regulatory analysis |
| Persona + Register | A legal advisor persona in executive register produces concise risk summaries |
| Domain constraint + Register | A product support constraint in conversational register produces warm, scoped customer responses |
| Domain constraint + Register | A product support constraint in professional register produces clear documentation-style answers |